In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet


In [ ]:
import pandas as pd

link = "https://huggingface.co/datasets/hamzas/nba-games/resolve/main/games.csv"
link2 = "https://huggingface.co/datasets/hamzas/nba-games/resolve/main/teams.csv"
data = pd.read_csv(link)



data.head()

In [12]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 29755 entries, 0 to 29754
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   GAME_DATE_EST     29755 non-null  str    
 1   GAME_ID           29755 non-null  int64  
 2   GAME_STATUS_TEXT  29755 non-null  str    
 3   HOME_TEAM_ID      29755 non-null  int64  
 4   VISITOR_TEAM_ID   29755 non-null  int64  
 5   SEASON            29755 non-null  int64  
 6   TEAM_ID_home      29755 non-null  int64  
 7   PTS_home          29656 non-null  float64
 8   FG_PCT_home       29656 non-null  float64
 9   FT_PCT_home       29656 non-null  float64
 10  FG3_PCT_home      29656 non-null  float64
 11  AST_home          29656 non-null  float64
 12  REB_home          29656 non-null  float64
 13  TEAM_ID_away      29755 non-null  int64  
 14  PTS_away          29656 non-null  float64
 15  FG_PCT_away       29656 non-null  float64
 16  FT_PCT_away       29656 non-null  float64
 17  FG3_

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt

# 1. Cargar y limpiar datos (como ya tenías)
link = "https://huggingface.co/datasets/hamzas/nba-games/resolve/main/games.csv"
data = pd.read_csv(link)
id_hawks = 1610612737
hawks = data[data['HOME_TEAM_ID'] == id_hawks].copy()
hawks['PTS_home'] = pd.to_numeric(hawks['PTS_home'], errors='coerce')
hawks = hawks.dropna(subset=['PTS_home']).reset_index(drop=True)

# 2. CREAR VARIABLES DE "MEMORIA" (Lags)
# Queremos que el modelo sepa qué pasó en los 3 partidos anteriores
for i in range(1, 4):
    hawks[f'lag_{i}'] = hawks['PTS_home'].shift(i)

# Eliminamos las filas donde no tenemos historia (las primeras 3)
hawks_ml = hawks.dropna().copy()

# Definimos X (los 3 partidos previos) y y (el partido actual)
X = hawks_ml[['lag_1', 'lag_2', 'lag_3']]
y = hawks_ml['PTS_home']

# 3. ENTRENAR UN MODELO MÁS DINÁMICO (Random Forest)
# El Random Forest maneja mejor la volatilidad que la Regresión Lineal
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

# 4. PREDICCIÓN "PASO A PASO"
# Para predecir el futuro, necesitamos usar la predicción anterior como dato
ultimos_datos = list(y.tail(3).values)
predicciones_futuras = []

for _ in range(10):
    input_data = np.array(ultimos_datos[-3:]).reshape(1, -1)
    pred = model.predict(input_data)[0]
    predicciones_futuras.append(pred)
    ultimos_datos.append(pred)

# 5. GRAFICAR RESULTADOS RECIENTES (Últimos 50 partidos + 10 predicciones)
plt.figure(figsize=(12, 6))
indices_reales = np.arange(len(y[-50:]))
plt.plot(indices_reales, y[-50:], 'ro-', label='Puntos Reales (Últimos 50)')

indices_pred = np.arange(len(y[-50:]), len(y[-50:]) + 10)
plt.plot(indices_pred, predicciones_futuras, 'b*--', label='Predicción Próximos 10')

plt.axvline(x=len(y[-50:])-1, color='gray', linestyle='--')
plt.title('Predicción Dinámica con Random Forest - Hawks')
plt.legend()
plt.grid(True)
plt.show()

for i, p in enumerate(predicciones_futuras, 1):
    print(f"Partido {i}: {p:.2f} puntos")

In [27]:
from nba_api.stats.endpoints import leaguegamefinder
import pandas as pd

# ID de los Boston Celtics
id_boston = 1610612738

# 1. EXTRACCIÓN corregida
# Usamos team_id_nullable para filtrar directamente por los Celtics
game_finder = leaguegamefinder.LeagueGameFinder(team_id_nullable=id_boston)
games = game_finder.get_data_frames()[0]

# 2. TRANSFORMACIÓN: Filtrar por la temporada actual (2025-26)
# El ID '22025' corresponde a la temporada regular 2025
current_season = games[games['SEASON_ID'] == '22025']

# Filtrar partidos de local (donde NO aparece '@' en el MATCHUP)
boston_home = current_season[~current_season['MATCHUP'].str.contains('@')].copy()

# Mostrar los datos más frescos para tus apuestas
print(boston_home[['GAME_DATE', 'MATCHUP', 'PTS']].head())

    GAME_DATE      MATCHUP  PTS
0  2026-03-01  BOS vs. PHI  114
1  2026-02-27  BOS vs. BKN  148
6  2026-02-11  BOS vs. CHI  124
7  2026-02-08  BOS vs. NYK   89
8  2026-02-06  BOS vs. MIA   98


In [61]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder

# 1. PREPARACIÓN DE DATOS BASE
df_home = boston_home.copy()
df_home['GAME_DATE'] = pd.to_datetime(df_home['GAME_DATE'])
df_home = df_home.sort_values('GAME_DATE', ascending=True)

# 2. INGENIERÍA DE CARACTERÍSTICAS (Calculamos todo ANTES de entrenar)
# Puntos del Rival (Calculado)
if 'OPP_PTS' not in df_home.columns:
    df_home['OPP_PTS'] = df_home['PTS'] - df_home['PLUS_MINUS']

# Lags (Memoria de Boston)
df_home['lag_1'] = df_home['PTS'].shift(1)
df_home['lag_2'] = df_home['PTS'].shift(2)
df_home['lag_3'] = df_home['PTS'].shift(3)

# Back-to-Back (Descanso)
df_home['DAYS_REST'] = df_home['GAME_DATE'].diff().dt.days
df_home['IS_B2B'] = (df_home['DAYS_REST'] == 1).astype(int)

# Identificación de Rival y Codificación
def extract_rival_id(matchup):
    return matchup.split('vs.')[1].strip() if 'vs.' in matchup else matchup.split('@')[1].strip()

df_home['RIVAL'] = df_home['MATCHUP'].apply(extract_rival_id)
le = LabelEncoder()
df_home['RIVAL_ID'] = le.fit_transform(df_home['RIVAL'])

# 3. FILTRADO Y ENTRENAMIENTO DE MODELOS
df_ml = df_home.dropna(subset=['PTS', 'lag_1', 'lag_2', 'lag_3']).copy()

features = ['RIVAL_ID', 'lag_1', 'lag_2', 'lag_3', 'IS_B2B']
X = df_ml[features]

# Modelo para puntos de Boston
model_bos = RandomForestRegressor(n_estimators=1000, random_state=40)
model_bos.fit(X, df_ml['PTS'])

# Modelo para puntos del Rival
model_riv = RandomForestRegressor(n_estimators=1000, random_state=40)
model_riv.fit(X, df_ml['OPP_PTS'])

# 4. FUNCIÓN DE SCOUTING COMPLETO (BOSTON + RIVAL)
def predecir_scouting_total(nombre_rival, es_b2b=0):
    try:
        rival_idx = le.transform([nombre_rival])[0]
        lags = df_home['PTS'].tail(3).values 
        
        entrada = pd.DataFrame([[rival_idx, lags[2], lags[1], lags[0], es_b2b]], 
                               columns=features)
        
        pred_bos = model_bos.predict(entrada)[0]
        pred_riv = model_riv.predict(entrada)[0]
        
        estado = "CANSADOS (B2B)" if es_b2b == 1 else "DESCANSADOS"
        
        print(f"--- Scouting Report: Celtics vs {nombre_rival} ---")
        print(f"Estado físico: {estado}")
        print(f"🏀 Puntos estimados Celtics: {pred_bos:.2f}")
        print(f"🛡️ Puntos estimados {nombre_rival}: {pred_riv:.2f}")
        print(f"🔥 TOTAL (Over/Under): {pred_bos + pred_riv:.2f}")
        print(f"📈 Margen: {pred_bos - pred_riv:.2f}")
        
    except ValueError:
        print(f"Error: El equipo {nombre_rival} no está en los registros.")

# --- PRUEBA PARA MAÑANA ---
predecir_scouting_total('MIA')

--- Scouting Report: Celtics vs MIA ---
Estado físico: DESCANSADOS
🏀 Puntos estimados Celtics: 118.64
🛡️ Puntos estimados MIA: 113.50
🔥 TOTAL (Over/Under): 232.15
📈 Margen: 5.14


In [ ]:
import matplotlib.pyplot as plt

# Ver qué variable es más importante para predecir los puntos de Boston
importancias = model_bos.feature_importances_
plt.barh(features, importancias)
plt.title("¿Qué influye más en los puntos de Boston?")
plt.show()